# 1) Imports

In [24]:
import json
import sys
import os
import pandas as pd
import random
import torch
from torch.utils.data import DataLoader

# Get the absolute path to the project root
project_root = os.path.dirname(os.path.dirname(os.getcwd()))

# Add correct paths
models_path = os.path.join(project_root, 'models')
src_path = os.path.join(project_root, 'src')
data_path = os.path.join(project_root, 'data')

# Append to sys.path\n",
sys.path.append(models_path)
sys.path.append(src_path)
sys.path.append(data_path)

from data_loader_helper import SequenceDataset, collate_batch
from DKT.dkt_k_fold import k_fold_cv_dkt
from DKT.dkt_train import train_dkt
from KTDataset import KTDataset


In [13]:
project_root = os.path.abspath(os.path.join(os.path.abspath('notebooks'), '..'))
project_root

'c:\\Users\\Botond\\Documents\\Uni\\UniWien\\Subjects\\2024W\\DAP\\personalized-education\\notebooks\\DKT'

In [15]:
os.path.dirname(os.path.abspath('notebooks'))

'c:\\Users\\Botond\\Documents\\Uni\\UniWien\\Subjects\\2024W\\DAP\\personalized-education\\notebooks\\DKT'

In [23]:
os.path.dirname(os.path.dirname(os.getcwd()))

'c:\\Users\\Botond\\Documents\\Uni\\UniWien\\Subjects\\2024W\\DAP\\personalized-education'

'c:\\Users\\Botond\\Documents\\Uni\\UniWien\\Subjects\\2024W\\DAP\\personalized-education\\notebooks\\DKT'

# 2) Model Training and Evaluation

## 2.1) Data imports and transforms


In [55]:
df_answers = pd.read_csv('preprocessed/df_answers.csv')
df_skill_names = pd.read_csv('preprocessed/df_skill_names.csv')

additional_columns = None
#additional_columns = ['ease']
#additional_columns = ['ease', 'ms_first_response', 'bottom_hint']

kt_dataset = KTDataset(df_answers, df_skill_names, prepare_DKT=True, additional_columns=additional_columns)

user_dict = kt_dataset.DKT_datadict
num_skills = kt_dataset.num_skills
num_other = kt_dataset.num_other


## 2.2) Seting constants

In [56]:
train_ratio = 0.8  # 80% for training, 20% for testing
NUM_EPOCHS = 15
BATCH_SIZE = 100
NUM_FOLDS = 5  # Number of folds for cross-validation
HID_SIZE = 200

embed_dim = 3
special_embed = "tanh"

finetune = True


In [ ]:
if additional_columns is None:
    model_version = "basic"
elif len(additional_columns) == 1:
    model_version = "ease"
else:
    model_version = "full"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 2.3) Splitting data into train and test sets

In [57]:
# Get all keys and shuffle them
keys = list(user_dict.keys())
random.shuffle(keys)

# Split keys into train and test
split_index = int(len(keys) * train_ratio)
train_keys = keys[:split_index]
test_keys = keys[split_index:]

# Create train and test dictionaries
train_dict = {key: user_dict[key] for key in train_keys}
test_dict = {key: user_dict[key] for key in test_keys}


## 2.4) Hyperparameter-tuning

In [ ]:
if finetune:
  configs = [
    {
        "learning_rate": lr,
        "model_params": {
            "num_skills": num_skills,
            "num_other": num_other,
            "embed_dim": embed_dim,
            "hid_size": HID_SIZE,
            "num_hid_layers": num_hid_layers,
            "drop_prob": drop_prob,
            "special_embed": special_embed
        },
    }
    for lr in [1e-3, 1e-4, 1e-5]  # 3 reasonable options for learning rate
    for num_hid_layers in [1, 2, 3]  # Hidden layers 1 or 2
    for drop_prob in [0.3, 0.4, 0.5]  # Dropout rate 0.3, 0.4, 0.5
    ]

  best_val_auc_avg = 0  # Track the best validation AUC
  best_config = None  # Track the best configuration

  for idx, config in enumerate(configs, 1):
      log_message = f"\nEvaluating Config {idx}/{len(configs)}"
      print(log_message)  # Print to console

      lr = config['learning_rate']
      model_params = config['model_params']

      val_auc_avg = k_fold_cv_dkt(
          num_folds=NUM_FOLDS,
          model_params=model_params,
          lr=lr,
          num_epochs=NUM_EPOCHS,
          device=device,
          train_dict=train_dict,
          batch_size=BATCH_SIZE,
          )

      # Update the global best if needed
      if val_auc_avg > best_val_auc_avg:
          best_val_auc_avg = val_auc_avg
          best_config = config  # Save the best configuration
          best_message = f"\nNew best model found: Config {idx}. Validation AUC: {best_val_auc_avg:.4f}"
          print(best_message)  # Print to console

  # Final output
  final_message = f"\nBest test AUC: {best_val_auc_avg:.4f}"
  print(final_message)

  final_config_message = f"\nBest Configuration: {best_config}"
  print(final_config_message)

else:
  # Load the configuration from the JSON file
  with open('dkt_best_config.json', 'r') as f:
      best_config = json.load(f)


Evaluating Config 1/27

Fold 1/5

Epoch 1

Evaluation: 100%|██████████| 6/6 [00:01<00:00,  4.45 batches/s]
For the 1. epoch AUC is 0.673652392761294, loss is 0.44927384952704114.

Epoch 2

Evaluation: 100%|██████████| 6/6 [00:01<00:00,  5.73 batches/s]
For the 2. epoch AUC is 0.6885692458078502, loss is 0.4425662060578664.

Epoch 3

Evaluation: 100%|██████████| 6/6 [00:01<00:00,  5.80 batches/s]
For the 3. epoch AUC is 0.6951292525495479, loss is 0.4392680327097575.

Epoch 4

Evaluation: 100%|██████████| 6/6 [00:01<00:00,  4.46 batches/s]
For the 4. epoch AUC is 0.6920046690645205, loss is 0.4375188648700714.

Epoch 5

Evaluation: 100%|██████████| 6/6 [00:01<00:00,  5.57 batches/s]
For the 5. epoch AUC is 0.6961868169987063, loss is 0.43640587230523425.

Epoch 6

Evaluation: 100%|██████████| 6/6 [00:01<00:00,  5.69 batches/s]
For the 6. epoch AUC is 0.7000555229164407, loss is 0.4353921363751094.

Epoch 7

Evaluation: 100%|██████████| 6/6 [00:01<00:00,  4.62 batches/s]
For the 7. epoc

## 2.5) Model training

In [48]:
# Training on the whole train set
train_dict = dict(sorted(train_dict.items(), key=lambda item: len(item[1])))
train_dataset = SequenceDataset(train_dict)
train_loader = DataLoader(train_dataset, batch_size=100, collate_fn=collate_batch, pin_memory=False, num_workers=2)

test_dict = dict(sorted(test_dict.items(), key=lambda item: len(item[1])))
test_dataset = SequenceDataset(test_dict)
test_loader = DataLoader(test_dataset, batch_size=100, collate_fn=collate_batch, pin_memory=False, num_workers=2)

# Log training details
training_message = "Training on the whole train set"
print(training_message)  # Print to console

best_model, list_val_loss, list_val_auc = train_dkt(
    model_params=best_config['model_params'],
    lr=best_config['learning_rate'],
    num_epochs=NUM_EPOCHS,
    device=device,
    train_loader=train_loader
)


Training on the whole train set

Epoch 1

Training: 100%|██████████| 30/30 [00:07<00:00,  4.07 batches/s]
No validation loader provided. Using training data for evaluation.
Evaluation: 100%|██████████| 30/30 [00:04<00:00,  6.76 batches/s]
For the 1. epoch AUC is 0.6962938881815839, loss is 0.5624951263268788.

Epoch 2

Training: 100%|██████████| 30/30 [00:05<00:00,  5.45 batches/s]
No validation loader provided. Using training data for evaluation.
Evaluation: 100%|██████████| 30/30 [00:05<00:00,  5.38 batches/s]
For the 2. epoch AUC is 0.7101586919037519, loss is 0.5553750862677892.

Epoch 3

Training: 100%|██████████| 30/30 [00:04<00:00,  6.41 batches/s]
No validation loader provided. Using training data for evaluation.
Evaluation: 100%|██████████| 30/30 [00:05<00:00,  5.33 batches/s]
For the 3. epoch AUC is 0.7071610067040799, loss is 0.5532138794660568.

Epoch 4

Training: 100%|██████████| 30/30 [00:05<00:00,  5.95 batches/s]
No validation loader provided. Using training data for ev

## 2.6) Model evaluation

In [50]:
train_loss, train_auc = process(best_model, train_loader, device)
test_loss, test_auc = process(best_model, test_loader, device)

# Example results for a model
results = {
    'Dataset': ['Train', 'Test'],
    'AUC': [train_auc, test_auc],
    'Loss': [train_loss, test_loss]
}

# Convert to DataFrame
df = pd.DataFrame(results)

print(df)


Evaluation: 100%|██████████| 8/8 [00:01<00:00,  4.91 batches/s]
  Dataset       AUC      Loss
0   Train  0.713485  0.547942
1    Test  0.713036  0.444262


In [54]:
train_loss, train_auc = process(best_model, train_loader, device)
test_loss, test_auc = process(best_model, test_loader, device)

# Example results for a model
results = {
    'Dataset': ['Train', 'Test'],
    'AUC': [train_auc, test_auc],
    'Loss': [train_loss, test_loss]
}

# Convert to DataFrame
df = pd.DataFrame(results)

print(df)


Evaluation: 100%|██████████| 8/8 [00:01<00:00,  5.91 batches/s]
  Dataset       AUC      Loss
0   Train  0.714369  0.546642
1    Test  0.711428  0.452316


## 2.7) Saving the results

In [ ]:
# Saving the model
model_save_path = f'dkt_{model_version}_model.pth'
torch.save(best_model.state_dict(), model_save_path)
model_save_message = f"Model saved to {model_save_path}"

print(model_save_message)  # Print to console

# Save as CSV
df.to_csv(f'dkt_{model_version}_model_results.csv', index=False)
